In [ ]:
# ===============================================================================
# 0. INSTALL
# ==============================================================================

!pip install -q -U unsloth
!pip install -q datasets transformers trl accelerate bitsandbytes
!pip install -q huggingface_hub
# ==============================================================================
# 1. IMPORTS
# ==============================================================================
import os
import torch
import pandas as pd
from datasets import Dataset, load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from huggingface_hub import notebook_login
# ==============================================================================
# 2. USER CONFIGURATION
# ==============================================================================

# ------------------------------------------------------------------------------
# Hugging Face model
# ------------------------------------------------------------------------------

MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507"

# Examples:
#
# MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
# MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
# MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507"
#
# Replace with your desired Hugging Face model.
# Model must be compatible with your installed Unsloth version.


# ------------------------------------------------------------------------------
# Dataset
# ------------------------------------------------------------------------------

DATASET_PATH = "/content/usajobs_tech_roles_2026.csv"

# Supported examples:
#
# CSV:
# /content/my_dataset.csv
#
# JSON:
# /content/my_dataset.json
#
# JSONL:
# /content/my_dataset.jsonl


# ------------------------------------------------------------------------------
# Output
# ------------------------------------------------------------------------------

OUTPUT_DIR = "/content/my_finetuned_model"

# Hugging Face repository
HF_REPO = "vikram2xx7/Qwen3-4B-Instruct-2507-model"


# ------------------------------------------------------------------------------
# Training
# ------------------------------------------------------------------------------

MAX_SEQ_LENGTH = 2048

LOAD_IN_4BIT = True

LORA_R = 16

LORA_ALPHA = 16

EPOCHS = 1

LEARNING_RATE = 2e-4

BATCH_SIZE = 2

GRADIENT_ACCUMULATION = 4


# ============================================================
# 3. CHECK GPU
# ============================================================

print("GPU:")
print(torch.cuda.get_device_name(0))

print(
    "CUDA:",
    torch.cuda.is_available()
)


# ============================================================
# 4. LOAD DATASET
# ============================================================

file_extension = DATASET_PATH.split(".")[-1].lower()


if file_extension == "csv":

    df = pd.read_csv(DATASET_PATH)

    print("Dataset loaded as CSV")


elif file_extension == "json":

    df = pd.read_json(DATASET_PATH)

    print("Dataset loaded as JSON")


elif file_extension == "jsonl":

    df = pd.read_json(
        DATASET_PATH,
        lines=True
    )

    print("Dataset loaded as JSONL")


else:

    raise ValueError(
        "Unsupported dataset format. "
        "Use CSV, JSON, or JSONL."
    )


print("\nDataset shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst rows:")
display(df.head())


# ============================================================
# 5. CONVERT YOUR DATASET TO TRAINING TEXT
# ============================================================
#
# IMPORTANT:
#
# CHANGE THIS FUNCTION according to your dataset.
#
# Your dataset can contain ANY columns.
#
# The model ultimately receives TEXT.
#
# ============================================================


def create_training_text(row):

    # --------------------------------------------------------
    # UNIVERSAL VERSION
    #
    # Automatically converts every column into text.
    # --------------------------------------------------------

    text = ""

    for column in df.columns:

        value = row[column]

        text += f"{column}: {value}\n"

    return text


df["text"] = df.apply(
    create_training_text,
    axis=1
)


# ------------------------------------------------------------
# Remove empty values
# ------------------------------------------------------------

df["text"] = df["text"].fillna("")


# ------------------------------------------------------------
# Convert Pandas → Hugging Face Dataset
# ------------------------------------------------------------

dataset = Dataset.from_pandas(
    df[["text"]],
    preserve_index=False
)


print("\nTraining dataset:")
print(dataset)

print("\nExample:")
print(dataset[0]["text"])


# ============================================================
# 6. LOAD HUGGING FACE MODEL
# ============================================================

print("\nLoading model...")

model, tokenizer = FastLanguageModel.from_pretrained(

    model_name=MODEL_NAME,

    max_seq_length=MAX_SEQ_LENGTH,

    load_in_4bit=LOAD_IN_4BIT,

)


# ============================================================
# 7. ADD LoRA
# ============================================================

print("\nAdding LoRA adapters...")


model = FastLanguageModel.get_peft_model(

    model,

    r=LORA_R,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    lora_alpha=LORA_ALPHA,

    lora_dropout=0,

    bias="none",

    use_gradient_checkpointing="unsloth",

    random_state=3407,

)


# ============================================================
# 8. TRAINER
# ============================================================

trainer = SFTTrainer(

    model=model,

    tokenizer=tokenizer,

    train_dataset=dataset,

    dataset_text_field="text",

    max_seq_length=MAX_SEQ_LENGTH,

    packing=True,

    args=SFTConfig(

        output_dir=OUTPUT_DIR,

        per_device_train_batch_size=BATCH_SIZE,

        gradient_accumulation_steps=GRADIENT_ACCUMULATION,

        num_train_epochs=EPOCHS,

        learning_rate=LEARNING_RATE,

        warmup_steps=5,

        logging_steps=10,

        save_steps=100,

        save_total_limit=2,

        optim="adamw_8bit",

        weight_decay=0.01,

        lr_scheduler_type="linear",

        fp16=not torch.cuda.is_bf16_supported(),

        bf16=torch.cuda.is_bf16_supported(),

        seed=3407,

        report_to="none",

    ),

)


# ============================================================
# 9. START FINE-TUNING
# ============================================================

print("\nStarting fine-tuning...")

trainer_stats = trainer.train()

print("\nTraining completed.")


# ============================================================
# 10. SAVE FINE-TUNED MODEL
# ============================================================

print("\nSaving model...")

model.save_pretrained(
    OUTPUT_DIR
)

tokenizer.save_pretrained(
    OUTPUT_DIR
)

print(
    f"Model saved to: {OUTPUT_DIR}"
)


# ============================================================
# 11. PREPARE FOR INFERENCE
# ============================================================

FastLanguageModel.for_inference(
    model
)


# ============================================================
# 12. TEST MODEL
# ============================================================

def test_model(question):

    prompt = f"""### Instruction:
{question}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")


    outputs = model.generate(

        **inputs,

        max_new_tokens=256,

        temperature=0.7,

        do_sample=True,

        top_p=0.9,

    )


    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


    return response


# ============================================================
# 13. RUN TEST
# ============================================================

question = "Explain the information contained in this dataset."

response = test_model(
    question
)

print("\n==============================")
print("MODEL RESPONSE")
print("==============================\n")

print(response)


# ============================================================
# 14. LOGIN TO HUGGING FACE
# ============================================================

print("\nLogin to Hugging Face...")

notebook_login()


# ============================================================
# 15. UPLOAD LoRA MODEL
# ============================================================

print("\nUploading model...")

model.push_to_hub(
    HF_REPO
)

tokenizer.push_to_hub(
    HF_REPO
)

print("\n================================")
print("UPLOAD COMPLETE")
print("================================")

print(
    f"https://huggingface.co/{HF_REPO}"
)

changing code

In [ ]:
"""

MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"

DATASET_PATH = "/content/my_dataset.csv"

HF_REPO = "YOUR_USERNAME/my-finetuned-model

"""